## Week 12 Lab · Big Data Tools (Google Colab Edition)

- **Student Name:** KWIZERA Pacifique
- **Student ID:** 28054
- **Program:** Software Engineering
- **Course:** Introduction to Big data
- **Notebook:** `Week12_KWIZERA_Pacifique.ipynb`



In [10]:
# STEP 1: Install PySpark (required at the start on Google Colab — takes ~2-3 minutes)
!pip install -q pyspark

In [11]:
import os

required_files = ["week12_transactions.csv", "week12_api_sample.json"]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"Warning: The following required files are missing from your directory: {', '.join(missing_files)}")
    print("Please upload them using the folder icon on the left panel.")
else:
    print("All required lab files are successfully detected! You can proceed to run the rest of the cells.")

All required lab files are successfully detected! You can proceed to run the rest of the cells.


In [12]:
import os
import pandas as pd
import json

# 1. Convert week12_transactions.xlsx to CSV if it exists
if os.path.exists('week12_transactions.xlsx'):
    print("Found 'week12_transactions.xlsx'. Converting it to 'week12_transactions.csv'...")
    excel_df = pd.read_excel('week12_transactions.xlsx')
    excel_df.to_csv('week12_transactions.csv', index=False)
    print("Successfully created 'week12_transactions.csv'!")

# 2. Ensure week12_api_sample.json exists
if not os.path.exists('week12_api_sample.json'):
    print("Creating fallback 'week12_api_sample.json'...")
    sample_api_data = {
        "hourly": {
            "time": [f"2023-11-20T{hour:02d}:00" for hour in range(24)],
            "temperature_2m": [20.0, 19.5, 19.0, 18.5, 18.0, 18.2, 19.0, 21.0, 23.0, 24.5, 25.5, 26.0, 26.5, 27.0, 26.8, 26.0, 25.0, 23.5, 22.0, 21.5, 21.0, 20.8, 20.5, 20.2]
        }
    }
    with open('week12_api_sample.json', 'w', encoding='utf-8') as f:
        json.dump(sample_api_data, f)
    print("Successfully created 'week12_api_sample.json'!")

# 3. Final Check
if os.path.exists('week12_transactions.csv') and os.path.exists('week12_api_sample.json'):
    print("Both files are now ready. You can run all your code cells successfully!")
else:
    print("Warning: Please make sure you have uploaded 'week12_transactions.xlsx' or 'week12_transactions.csv' into the files tab.")

Found 'week12_transactions.xlsx'. Converting it to 'week12_transactions.csv'...
Successfully created 'week12_transactions.csv'!
Both files are now ready. You can run all your code cells successfully!


In [13]:
# STEP 3: Core imports
import sqlite3
import json
import requests
import pandas as pd
from pyspark.sql import SparkSession

pd.set_option("display.max_columns", None)
print("All libraries imported successfully.")

All libraries imported successfully.


---
## Part 1 · SQL with Python

Load `week12_transactions.csv` into an SQLite database (`momo.db`, table `transactions`).

In [14]:
# Load CSV into SQLite database 'momo.db', table 'transactions'
df_txn = pd.read_csv("week12_transactions.csv")
conn = sqlite3.connect("momo.db")
df_txn.to_sql("transactions", conn, if_exists="replace", index=False)

print(f"Loaded {len(df_txn):,} rows into momo.db (table: transactions).")
df_txn.head()

Loaded 2,000 rows into momo.db (table: transactions).


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,46327,8,Gasabo,send,1000,0
1,T00002,46335,12,Nyarugenge,send,25000,0
2,T00003,46345,8,Muhanga,send,51000,0
3,T00004,46327,17,Huye,send,79000,0
4,T00005,46341,12,Huye,send,109000,0


### Exercise 1.1 — SELECT / WHERE / ORDER BY
- **(a)** `txn_id`, `district`, `amount_rwf` of the 5 largest `send` transactions above 100,000 RWF.
- **(b)** Count the flagged transactions (`flagged = 1`). *(Checkpoint: 11 flagged)*

In [15]:
# (a) 5 largest 'send' transactions above 100,000 RWF
query_1_1a = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5;
"""
print("(a) 5 largest 'send' transactions above 100,000 RWF:")
display(pd.read_sql(query_1_1a, conn))

# (b) Count flagged transactions (Checkpoint: 11 flagged)
query_1_1b = """
SELECT COUNT(*) AS flagged_count
FROM transactions
WHERE flagged = 1;
"""
print("(b) Flagged transactions count:")
display(pd.read_sql(query_1_1b, conn))

(a) 5 largest 'send' transactions above 100,000 RWF:


,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


(b) Flagged transactions count:


,flagged_count
0,11


### Exercise 1.2 — GROUP BY
- **(a)** Per `txn_type`: `COUNT(*)`, `SUM` and `AVG` of `amount_rwf`, ordered by total descending — which type moves the most money?
- **(b)** Total per `district` — which district leads?
*(Checkpoint: `'send'` moves ~62.0M RWF; `Muhanga` leads with ~21.1M RWF)*

In [16]:
# (a) Per txn_type summary ordered by total descending
query_1_2a = """
SELECT
    txn_type,
    COUNT(*) AS txn_count,
    SUM(amount_rwf) AS total_amount_rwf,
    ROUND(AVG(amount_rwf), 2) AS avg_amount_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_amount_rwf DESC;
"""
df_by_type = pd.read_sql(query_1_2a, conn)
print("(a) Totals and averages by txn_type:")
display(df_by_type)

# (b) Total per district ordered descending
query_1_2b = """
SELECT
    district,
    COUNT(*) AS txn_count,
    SUM(amount_rwf) AS total_amount_rwf
FROM transactions
GROUP BY district
ORDER BY total_amount_rwf DESC;
"""
df_by_district = pd.read_sql(query_1_2b, conn)
print("(b) Total value by district:")
display(df_by_district)

# Comment:
# - 'send' moves the most money (~62.0M RWF).
# - Muhanga leads all districts (~21.1M RWF).

(a) Totals and averages by txn_type:


,txn_type,txn_count,total_amount_rwf,avg_amount_rwf
0,send,868,62049000,71485.02
1,paybill,330,29970000,90818.18
2,receive,489,27024000,55263.80
3,airtime,313,833000,2661.34


(b) Total value by district:


,district,txn_count,total_amount_rwf
0,Muhanga,247,21076000
1,Gasabo,255,20738000
2,Huye,251,18736000
3,Nyarugenge,252,18091000
4,Kicukiro,249,11338000
5,Musanze,253,10490500
6,Rubavu,246,10110500
7,Nyagatare,247,9296000


### Exercise 1.3 — The busiest hour
- **(a)** In SQL: transactions per `hour`, busiest first.
- **(b)** Answer the same question in Pandas and confirm both agree. *(Checkpoint: hour 20, with 185 transactions)*

In [17]:
# (a) SQL: transactions per hour, busiest first
query_1_3 = """
SELECT
    hour,
    COUNT(*) AS txn_count
FROM transactions
GROUP BY hour
ORDER BY txn_count DESC;
"""
hourly_sql = pd.read_sql(query_1_3, conn)
print("(a) Top 5 busiest hours (SQL):")
display(hourly_sql.head())

# (b) Pandas: transactions per hour, busiest first
hourly_pandas = (
    df_txn.groupby("hour")
    .size()
    .reset_index(name="txn_count")
    .sort_values(by="txn_count", ascending=False)
    .reset_index(drop=True)
)
print("(b) Top 5 busiest hours (Pandas):")
display(hourly_pandas.head())

# Confirm both agree
sql_top = hourly_sql.iloc[0]
pd_top = hourly_pandas.iloc[0]
assert int(sql_top["hour"]) == int(pd_top["hour"]) and int(sql_top["txn_count"]) == int(pd_top["txn_count"])
print(f"Confirmed: Both SQL and Pandas agree — hour {int(sql_top['hour'])} is busiest with {int(sql_top['txn_count'])} transactions.")

(a) Top 5 busiest hours (SQL):


,hour,txn_count
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


(b) Top 5 busiest hours (Pandas):


,hour,txn_count
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


Confirmed: Both SQL and Pandas agree — hour 20 is busiest with 185 transactions.


---
## Part 2 · APIs & JSON

### Exercise 2.1 — JSON by hand
Parse the JSON string with `json.loads` and print: **(a)** the city, **(b)** the number of readings, **(c)** their average (2 decimals).

In [18]:
# Sample JSON string (swap in the instructor's exact string if provided in class)
json_str = '{"city": "Kigali", "readings": [21.4, 23.1, 25.0, 26.2, 24.5, 22.1]}'

parsed = json.loads(json_str)
city = parsed["city"]
readings = parsed["readings"]
avg_reading = sum(readings) / len(readings)

print(f"(a) City:               {city}")
print(f"(b) Number of readings: {len(readings)}")
print(f"(c) Average reading:    {avg_reading:.2f}")

(a) City:               Kigali
(b) Number of readings: 6
(c) Average reading:    23.72


### Exercise 2.2 — Call a real API
Fetch Kigali's 24-hour forecast from Open-Meteo (falling back to `week12_api_sample.json` offline). Extract `temperature` and `time` lists and print:
- **(a)** how many hours,
- **(b)** max and min temperature,
- **(c)** the TIME of the hottest hour using `.index(max(...))`.

In [19]:
api_url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-1.9441&longitude=30.0619&hourly=temperature_2m&forecast_days=1"
)

try:
    resp = requests.get(api_url, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    print("Loaded live forecast from Open-Meteo API.")
except Exception as e:
    print(f"API call failed ({e}). Falling back to week12_api_sample.json...")
    with open("week12_api_sample.json", "r", encoding="utf-8") as f:
        data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

max_temp = max(temps)
min_temp = min(temps)
hottest_time = times[temps.index(max_temp)]

print(f"(a) Hours in forecast: {len(temps)}")
print(f"(b) Max temperature:   {max_temp:.1f} °C | Min temperature: {min_temp:.1f} °C")
print(f"(c) Hottest hour time: {hottest_time}")

Loaded live forecast from Open-Meteo API.
(a) Hours in forecast: 24
(b) Max temperature:   24.2 °C | Min temperature: 16.1 °C
(c) Hottest hour time: 2026-11-23T10:00


### Exercise 2.3 — API data meets your toolbox
Build a DataFrame with columns `time` and `temp_c` from the API data.
- **(a)** `.describe()` the temperature.
- **(b)** Count hours $\ge 25\text{ }^\circ\text{C}$ with a boolean mask.

In [20]:
weather_df = pd.DataFrame({"time": times, "temp_c": temps})

# (a) describe() on temperature
print("(a) Temperature summary statistics:")
print(weather_df["temp_c"].describe())

# (b) Count hours >= 25 °C using a boolean mask
hot_hours = (weather_df["temp_c"] >= 25).sum()
print(f"\n(b) Hours >= 25 °C: {hot_hours}")

(a) Temperature summary statistics:
count    24.000000
mean     19.420833
std       2.635705
min      16.100000
25%      16.925000
50%      19.000000
75%      21.675000
max      24.200000
Name: temp_c, dtype: float64

(b) Hours >= 25 °C: 0


---
## Part 3 · PySpark

Start a local `SparkSession` (startup warnings are normal).

In [21]:
spark = (
    SparkSession.builder
    .appName("AUCA_Week12_BigDataTools")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession running (version {spark.version})")

SparkSession running (version 4.0.5)


### Exercise 3.1 — Load & inspect
Load the CSV into a Spark DataFrame `sdf` with `header=True, inferSchema=True`. Print row count, schema, and first 5 rows.

In [22]:
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)

print(f"Row count: {sdf.count():,}")
print("\nSchema:")
sdf.printSchema()
print("First 5 rows:")
sdf.show(5)

Row count: 2,000

Schema:
root
 |-- txn_id: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

First 5 rows:
+------+-----+----+----------+--------+----------+-------+
|txn_id| date|hour|  district|txn_type|amount_rwf|flagged|
+------+-----+----+----------+--------+----------+-------+
|T00001|46327|   8|    Gasabo|    send|      1000|      0|
|T00002|46335|  12|Nyarugenge|    send|     25000|      0|
|T00003|46345|   8|   Muhanga|    send|     51000|      0|
|T00004|46327|  17|      Huye|    send|     79000|      0|
|T00005|46341|  12|      Huye|    send|    109000|      0|
+------+-----+----+----------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.2 — Filter
- **(a)** Count transactions above 100,000 RWF with `sdf.filter(...)`. *(Checkpoint: 451 transactions above 100k)*
- **(b)** Show 5 flagged transactions.

In [23]:
# (a) Count transactions > 100,000 RWF
above_100k = sdf.filter(sdf["amount_rwf"] > 100000).count()
print(f"(a) Transactions above 100,000 RWF: {above_100k} (Checkpoint: 451)")

# (b) Show 5 flagged transactions
print("\n(b) 5 flagged transactions:")
sdf.filter(sdf["flagged"] == 1).show(5)

(a) Transactions above 100,000 RWF: 451 (Checkpoint: 451)

(b) 5 flagged transactions:
+------+-----+----+--------+--------+----------+-------+
|txn_id| date|hour|district|txn_type|amount_rwf|flagged|
+------+-----+----+--------+--------+----------+-------+
|T00279|46355|  19| Muhanga|    send|    127000|      1|
|T00326|46333|  22|    Huye|    send|    127000|      1|
|T00536|46353|  17|  Gasabo|    send|    127000|      1|
|T00655|46356|  17|  Rubavu|    send|    134000|      1|
|T00697|46327|   9| Muhanga|    send|    127000|      1|
+------+-----+----+--------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.3 — groupBy, fourth reunion
- **(a)** Total amount per `txn_type` with `groupBy(...).sum(...)` and `.show()`.-
- **(b)** Per-district totals converted with `.toPandas()` and sorted.
- **(c)** Confirm in a comment that Spark answers match SQL answers.

In [24]:
# (a) Total amount per txn_type
print("(a) Total amount per txn_type (PySpark):")
sdf.groupBy("txn_type").sum("amount_rwf").orderBy("sum(amount_rwf)", ascending=False).show()

# (b) Per-district totals converted to Pandas and sorted descending
district_spark = (
    sdf.groupBy("district")
    .sum("amount_rwf")
    .toPandas()
    .rename(columns={"sum(amount_rwf)": "total_amount_rwf"})
    .sort_values(by="total_amount_rwf", ascending=False)
    .reset_index(drop=True)
)
print("(b) Total amount per district (PySpark -> Pandas sorted):")
display(district_spark)

# (c) Comment:
# Yes — the Spark answers match our Part 1 SQL answers identically ('send' moves ~62.0M RWF
# and Muhanga leads with ~21.1M RWF). Both engines execute the same deterministic aggregations.

(a) Total amount per txn_type (PySpark):
+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
|    send|       62049000|
| paybill|       29970000|
| receive|       27024000|
| airtime|         833000|
+--------+---------------+

(b) Total amount per district (PySpark -> Pandas sorted):


,district,total_amount_rwf
0,Muhanga,21076000
1,Gasabo,20738000
2,Huye,18736000
3,Nyarugenge,18091000
4,Kicukiro,11338000
5,Musanze,10490500
6,Rubavu,10110500
7,Nyagatare,9296000


### Bonus (+3) — Spark speaks SQL too
Register the temporary view `transactions` and run the Part 1 `GROUP BY` query unchanged via `spark.sql(...)`.

In [25]:
sdf.createOrReplaceTempView("transactions")

# Part 1 GROUP BY query executed unchanged in Spark SQL
spark.sql("""
SELECT
    txn_type,
    COUNT(*) AS txn_count,
    SUM(amount_rwf) AS total_amount_rwf,
    ROUND(AVG(amount_rwf), 2) AS avg_amount_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_amount_rwf DESC
""").show()

# Comment:
# Running the exact same SQL on SQLite and on a thousand-machine Spark cluster is powerful because
# SQL is declarative: it separates what you want to compute from how the hardware executes it.
# Engineers can prototype locally on megabytes in SQLite and deploy the identical query at petabyte
# scale on Spark without rewriting business logic or retraining analysts.

+--------+---------+----------------+--------------+
|txn_type|txn_count|total_amount_rwf|avg_amount_rwf|
+--------+---------+----------------+--------------+
|    send|      868|        62049000|      71485.02|
| paybill|      330|        29970000|      90818.18|
| receive|      489|        27024000|       55263.8|
| airtime|      313|          833000|       2661.34|
+--------+---------+----------------+--------------+



---
## Part 4 · Choose Your Weapon

1. **Ministry of Health gives you read-access to their PostgreSQL database and asks for monthly totals per district.**  
   **Tool:** **SQL** — Running `GROUP BY` directly inside PostgreSQL pushes the aggregation to the database server where the data already lives, using indexes and avoiding unnecessary data movement.

2. **Your project group has a 40,000-row CSV needing cleaning, three charts, and a regression model.**  
   **Tool:** **Pandas** — A 40,000-row CSV fits effortlessly in local RAM (a few megabytes), making Pandas the fastest, lowest-overhead environment for cleaning, visualization, and preparing features for scikit-learn.

3. **A telecom wants nationwide call-record analysis — 40 TB already on their cluster.**  
   **Tool:** **Spark** — At 40 terabytes, the dataset far exceeds single-machine memory and storage limits, requiring Spark's distributed, fault-tolerant parallel processing across the cluster.

4. **Your dashboard must always show today's weather without anyone uploading a file.**  
   **Tool:** **API** — Calling a live REST weather API over HTTP fetches fresh JSON data automatically on demand with zero manual file uploads.